# PlanForge
> Your AI business advisor. Tell it what you want to achieve, have a conversation, get a plan.

*v0.4.0*

In [ ]:
# @title Setup
!pip install -q pysolvr-client
import sys, json as _json
from google.colab import userdata, drive, files as colab_files
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    display(HTML('<div style="color:#f87171;padding:12px;border:1px solid #f87171;border-radius:8px;font-family:Inter,sans-serif"><b>API key not found</b><ol><li>Click key icon in left sidebar</li><li>Add: PLANFORGE_API_KEY</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'PLANFORGE_API_KEY not found'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'exports', 'uploads'])
drive_mgr.ensure_folders()

# Session state
_state = {'goal': None, 'history': [], 'locked': {}}

if client.health_check():
    ui.success('Connected', f'Drive: {drive_mgr.root}')
else:
    ui.error('Connection failed', 'Check API key')


In [ ]:
# @title What do you want to achieve?
GOAL = 'I need funding for my startup'  # @param ["I need funding for my startup", "I need a UK visa for my business", "I want to validate my idea", "I need a bank loan", "I'm applying to an accelerator", "I need a grant (Innovate UK)", "I just want a quick viability check"]

_goal_map = {
    'I need funding for my startup': 'seed_pitch',
    'I need a UK visa for my business': 'innovator_visa',
    'I want to validate my idea': 'validate',
    'I need a bank loan': 'bank_loan',
    "I'm applying to an accelerator": 'accelerator',
    'I need a grant (Innovate UK)': 'innovate_uk',
    'I just want a quick viability check': 'scorecard',
}
_state['goal'] = _goal_map.get(GOAL, 'scorecard')
_state['history'] = []
_state['locked'] = {}

# Fetch what topics are needed for this goal
result = client.call('GET', '/goals')
if result['ok']:
    goals = {g['id']: g for g in result['data']['goals']}
    goal_info = goals.get(_state['goal'], {})
    topics_needed = goal_info.get('required_topics', [])
    ui.card(f'Goal: {goal_info.get("name", GOAL)}',
            f'<b>Audience:</b> {goal_info.get("audience", "you")}<br>'
            f'<b>Topics to cover:</b> {", ".join(topics_needed)}<br><br>'
            f'Run the next cell to start your discovery conversation.',
            status='success')
else:
    ui.success(f'Goal set: {GOAL}', 'Run the next cell to start chatting')


In [ ]:
# @title Discovery Chat
# Run this cell once - the chat interface appears below.

_chat_out = widgets.Output(layout={'border': '1px solid #334155', 'border_radius': '12px',
                                    'padding': '16px', 'max_height': '500px',
                                    'overflow_y': 'auto', 'background_color': '#0f172a'})
_input_area = widgets.Textarea(placeholder='Type your message, or paste text (CV, financials, notes)...',
                               layout=widgets.Layout(width='100%', height='80px'))
_upload_btn = widgets.FileUpload(accept='.txt,.pdf,.csv,.md,.doc,.docx', multiple=False,
                                  description='Attach file')
_send_btn = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='100px'))
_status_out = widgets.Output()


def _render_chat():
    with _chat_out:
        clear_output(wait=True)
        if not _state['history']:
            display(HTML('<p style="color:#94a3b8;font-family:Inter,sans-serif;font-size:14px">' 
                         'Tell me about your business. You can type a message, paste a document, '
                         'or upload a file (CV, financials, notes).</p>'))
            return
        html = ''
        for msg in _state['history']:
            if msg['role'] == 'user':
                text = msg['content'][:500] + ('...' if len(msg['content']) > 500 else '')
                html += (f'<div style="text-align:right;margin:8px 0">' 
                         f'<span style="background:#1e40af;color:#f1f5f9;padding:8px 14px;'
                         f'border-radius:16px 16px 4px 16px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{text}</span></div>')
            else:
                html += (f'<div style="text-align:left;margin:8px 0">' 
                         f'<span style="background:#1e293b;color:#e2e8f0;padding:8px 14px;'
                         f'border-radius:16px 16px 16px 4px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{msg["content"]}</span></div>')
        display(HTML(html))


def _on_send(btn):
    message = _input_area.value.strip()
    # Check for file upload
    if _upload_btn.value:
        for fname, fdata in _upload_btn.value.items():
            content = fdata['content'].decode('utf-8', errors='replace')
            drive_mgr.save('uploads', fname, content)
            message = f'[Attached: {fname}]\n\n{content}' + ('\n\n' + message if message else '')
        _upload_btn.value = {}
    if not message:
        with _status_out:
            clear_output(wait=True)
            display(HTML('<span style="color:#f87171;font-size:12px">Type a message or attach a file</span>'))
        return
    _state['history'].append({'role': 'user', 'content': message})
    _input_area.value = ''
    _render_chat()
    with _status_out:
        clear_output(wait=True)
        display(HTML('<span style="color:#94a3b8;font-size:12px">Thinking...</span>'))
    # Determine current topic from profile
    profile_result = client.call('GET', '/profile')
    locked = {}
    if profile_result['ok']:
        topics = profile_result['data'].get('topics', {})
        locked = {k: v.get('summary', '') for k, v in topics.items() if v.get('locked')}
    _state['locked'] = locked
    # Pick next unlocked topic
    result = client.call('GET', '/goals')
    topic = 'business_overview'
    if result['ok']:
        goals = {g['id']: g for g in result['data']['goals']}
        goal_info = goals.get(_state['goal'], {})
        for t in goal_info.get('required_topics', []):
            if t not in locked:
                topic = t
                break
    # Send to API
    body = {'topic': topic, 'message': message, 'goal_type': _state['goal'],
            'conversation_history': [{'role': m['role'], 'content': m['content'][:1000]} for m in _state['history'][-6:]],
            'locked_topics': locked}
    resp = client.run_async('/chat', body)
    with _status_out:
        clear_output(wait=True)
    if resp['ok']:
        reply = resp['data'].get('response', resp['data'].get('reply', ''))
        _state['history'].append({'role': 'assistant', 'content': reply})
        if resp['data'].get('topic_status') == 'locked':
            with _status_out:
                display(HTML(f'<span style="color:#10b981;font-size:12px">Topic locked! Moving to next.</span>'))
    else:
        _state['history'].append({'role': 'assistant', 'content': f"Error: {resp.get('error', 'Something went wrong')}"})
    _render_chat()


_send_btn.on_click(_on_send)
_render_chat()
display(widgets.VBox([
    _chat_out,
    widgets.HBox([_input_area, widgets.VBox([_send_btn, _upload_btn])]),
    _status_out,
]))


In [ ]:
# @title Am I ready to generate?
result = client.call('POST', '/readiness', {'goal_type': _state['goal']})
if result['ok']:
    data = result['data']
    emoji = {'ready': '\u2705', 'almost': '\u26a0\ufe0f', 'not_ready': '\u274c'}.get(data['verdict'], '')
    ui.card(f"{emoji} Readiness: {data['score']}/100",
            f"<b>Verdict:</b> {data['verdict']}<br><b>Next:</b> {data['recommended_next']}",
            status='success' if data['verdict'] == 'ready' else 'warning')
    if data.get('gaps'):
        rows = [{'Topic': g['topic'], 'Action': g['action'], 'Priority': g['priority']} for g in data['gaps']]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Readiness check failed'))


In [ ]:
# @title Generate my plan
result = client.run_async('/generate', {'goal_type': _state['goal']})
if result['ok']:
    data = result['data']
    plan_id = data.get('plan_id', '')
    content = data.get('content', data.get('plan', ''))
    ui.card('Your Plan', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{content}</pre>', status='success')
    if plan_id:
        filename = f'{_state["goal"]}_{plan_id[:8]}.md'
        drive_mgr.save('plans', filename, content, meta={'plan_id': plan_id, 'goal': _state['goal']})
        ui.success('Saved to Drive', f'plans/{filename}')
else:
    ui.error(result.get('error', 'Generation failed'), f'Status: {result.get("status")}')


In [ ]:
# @title Export to a different format
FORMAT = 'executive_summary'  # @param ["executive_summary", "pitch_deck", "one_pager", "financial_summary", "full_pdf"]

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    result = client.run_async('/export', {'content': content, 'format': FORMAT})
    if result['ok']:
        data = result['data']
        ui.card(f'Export: {FORMAT}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
        filename = f'{FORMAT}_{plan_files[-1].replace(".md", "")}.md'
        drive_mgr.save('exports', filename, data['content'], meta={'format': FORMAT})
        ui.success('Saved', f'exports/{filename}')
    else:
        ui.error(result.get('error', 'Export failed'))


In [ ]:
# @title Refine a section
SECTION = ''  # @param {type:"string"}
FEEDBACK = ''  # @param {type:"string"}

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
elif not SECTION.strip():
    ui.error('Enter a section name', 'e.g. Market Analysis, Financial Projections')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    body = {'content': content, 'section': SECTION}
    if FEEDBACK.strip():
        body['feedback'] = FEEDBACK
    result = client.run_async('/refine', body)
    if result['ok']:
        data = result['data']
        ui.card(f'Refined: {data["section"]}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
    else:
        ui.error(result.get('error', 'Refinement failed'))


In [ ]:
# @title My plans
result = client.call('GET', '/plans')
if result['ok']:
    plans = result['data'].get('plans', [])
    if not plans:
        ui.card('Plans', 'No plans generated yet.')
    else:
        rows = [{'ID': p['plan_id'][:8], 'Goal': p.get('goal_type', ''), 'Created': p.get('created_at', '')[:10]} for p in plans]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch plans'))


In [ ]:
# @title Account
result = client.call('GET', '/account')
if result['ok']:
    d = result['data']
    ui.card('Account', (
        f"<b>Plan:</b> {d.get('plan', 'free')}<br>"
        f"<b>Spend:</b> ${d.get('monthly_spend_usd', 0):.2f} / ${d.get('monthly_limit_usd', 0):.2f}<br>"
        f"<b>Top-up:</b> ${d.get('topup_balance_usd', 0):.2f}"
    ), status='success')
else:
    ui.error(result.get('error', 'Could not fetch account'))


In [ ]:
# @title Usage
result = client.get_usage()
if result['ok']:
    data = result['data']
    ui.usage_bar(data.get('current_month_spend_usd', 0), data.get('monthly_limit_usd', 1))
    if data.get('recent'):
        ui.table(data['recent'])
else:
    ui.error(result.get('error', 'Could not fetch usage'), 'Check your API key')
